In [1]:
# Import Librairies 
import pandas as pd
import os

In [2]:
# Vérifions que nous travaillons bien dans le venv
import sys
print(sys.executable)

E:\Workspace\Memory\.venv\Scripts\python.exe


In [3]:
# Chemin d'accès au data
DATA_PATH = "../data/raw/"

In [4]:
# vérification rapide des datasets :
files = [
    "CEAS_08.csv",
    "emails.csv",
    "Enron.csv",
    "Ling.csv",
    "Nazario.csv",
    "Nigerian_Fraud.csv",
    "phishing_email.csv",
    "SpamAssasin.csv"
]

for f in files:
    
    try:
        df = pd.read_csv(DATA_PATH + f, nrows=5)
        print("\nFILE :", f)
        print(df.columns)
        
    except Exception as e:
        print("Erreur avec", f, ":", e)


FILE : CEAS_08.csv
Index(['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'], dtype='str')

FILE : emails.csv
Index(['file', 'message'], dtype='str')

FILE : Enron.csv
Index(['subject', 'body', 'label'], dtype='str')

FILE : Ling.csv
Index(['subject', 'body', 'label'], dtype='str')

FILE : Nazario.csv
Index(['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label'], dtype='str')

FILE : Nigerian_Fraud.csv
Index(['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label'], dtype='str')

FILE : phishing_email.csv
Index(['text_combined', 'label'], dtype='str')

FILE : SpamAssasin.csv
Index(['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'], dtype='str')


In [5]:
# Fonctions de préparation :
def prepare_labeled_dataset(file, text_column):

    df = pd.read_csv(DATA_PATH + file)

    df = df[[text_column, "label"]].copy()

    df.columns = ["text", "label"]

    return df

    
def prepare_unlabeled_dataset(file, text_column, label):

    df = pd.read_csv(DATA_PATH + file)

    df = df[[text_column]].copy()

    df["label"] = label

    df.columns = ["text", "label"]

    return df


In [6]:
# Chargement des datasets :

# Ham :
emails = pd.read_csv(
    DATA_PATH + "emails.csv",
    usecols=["message"],
    nrows=50000
)

emails["label"] = "ham"

emails.columns = ["text", "label"]

enron = prepare_unlabeled_dataset("Enron.csv","body","ham")

# Spam :
ling = prepare_unlabeled_dataset("Ling.csv","body","spam")

spamass = prepare_unlabeled_dataset("SpamAssasin.csv","body","spam")

# Phishing :
ceas = prepare_unlabeled_dataset("CEAS_08.csv","body","phishing")

nazario = prepare_unlabeled_dataset("Nazario.csv","body","phishing")

nigerian = prepare_unlabeled_dataset("Nigerian_Fraud.csv","body","phishing")

phishing = prepare_unlabeled_dataset("phishing_email.csv","text_combined","phishing")


In [7]:
# Fusion des datasets :
datasets = [
    emails,
    enron,
    ling,
    spamass,
    ceas,
    nazario,
    nigerian,
    phishing
]

dataset = pd.concat(datasets)

In [8]:
# Nettoyage :
dataset = dataset.dropna()

dataset = dataset.drop_duplicates()

dataset = dataset.reset_index(drop=True)

In [9]:
# Conversion des types :
dataset["text"] = dataset["text"].astype(str)

dataset["label"] = dataset["label"].astype(str)

In [10]:
# Normalisation des labels :
# dataset["label"] = dataset["label"].replace({
#    "0": "ham",
#    "1": "spam",
#    0: "ham",
#    1: "spam"
#})

In [11]:
# Vérifier la distribution :
print(dataset["label"].value_counts())

print("\nTaille du dataset :", dataset.shape)

label
phishing    126128
ham          79767
spam          8667
Name: count, dtype: int64

Taille du dataset : (214562, 2)


In [12]:
# Sauvegarde :
dataset.to_csv("../data/dataset_final.csv", index=False)